In [8]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, f1_score

train = pd.read_csv("../data/raw/train.csv")
test = pd.read_csv("../data/raw/test.csv")


In [9]:
def model_olustur():
    return make_pipeline(
        TfidfVectorizer(
            analyzer="char_wb",     # kelime değil, karakter parçalarına bak
            ngram_range=(2, 4),     # 2-4 karakterlik parçalar: "gü", "güz", "güze"
            max_features=200_000,   # en sık 200 bin parçayı kullan
            min_df=2,               # sadece 1 kez geçen parçaları at
            sublinear_tf=True,
        ),
        LogisticRegression(max_iter=1000, C=4),
    )

In [10]:
model_a = model_olustur()
model_a.fit(train.text, train.label)

tahmin = model_a.predict(test.text)
print(classification_report(test.label, tahmin, digits=3))

              precision    recall  f1-score   support

    Negative      0.854     0.749     0.798      5656
        Notr      0.995     0.996     0.996     17092
    Positive      0.946     0.970     0.958     26217

    accuracy                          0.954     48965
   macro avg      0.932     0.905     0.917     48965
weighted avg      0.952     0.954     0.953     48965



In [11]:
def bosluk_duzelt(metinler):
    # \s+ = bir veya daha fazla boşluk,  ([.,!?;:]) = bir noktalama işareti
    # "boşluk + noktalama" gördüğün yerde sadece noktalamayı bırak
    return metinler.str.replace(r"\s+([.,!?;:])", r"\1", regex=True).str.strip()

test_duzeltilmis = bosluk_duzelt(test.text)
print(test.text[test.label == "Notr"].iloc[0])
print(test_duzeltilmis[test.label == "Notr"].iloc[0])

tahmin = model_a.predict(test_duzeltilmis)
print(classification_report(test.label, tahmin, digits=3))

Kral akbaba dikkat çekici renklere sahiptir .
Kral akbaba dikkat çekici renklere sahiptir.
              precision    recall  f1-score   support

    Negative      0.747     0.749     0.748      5656
        Notr      0.991     0.390     0.560     17092
    Positive      0.696     0.971     0.811     26217

    accuracy                          0.743     48965
   macro avg      0.811     0.703     0.706     48965
weighted avg      0.805     0.743     0.716     48965



In [12]:
model_b = model_olustur()
model_b.fit(bosluk_duzelt(train.text), train.label)

for ad, metin in [("normal test", bosluk_duzelt(test.text)), ("stres testi", test_duzeltilmis)]:
    tahmin = model_b.predict(metin)
    print(ad, "macro F1:", round(f1_score(test.label, tahmin, average="macro"), 3))

normal test macro F1: 0.912
stres testi macro F1: 0.912


In [13]:
cumleler = [
    "Kargo bugün geldi.",
    "Toplantı saat üçte başlıyor.",
    "Ürün mavi renkte.",
    "Bu ürünü dün aldım.",
    "Fiyatı 200 TL.",
    "Paket kapıya bırakıldı.",
]
pd.DataFrame({
    "cumle": cumleler,
    "model_a": model_a.predict(cumleler),
    "model_b": model_b.predict(cumleler),
})

,cumle,model_a,model_b
0,Kargo bugün geldi.,Positive,Positive
1,Toplantı saat üçte başlıyor.,Negative,Notr
2,Ürün mavi renkte.,Positive,Positive
3,Bu ürünü dün aldım.,Positive,Positive
4,Fiyatı 200 TL.,Positive,Positive
5,Paket kapıya bırakıldı.,Positive,Notr
